In [110]:
# ============================================================
# MyGPT2 - Complete Checkpoint Evaluation
# ============================================================

from pathlib import Path
import sys
import json
import math
import re
import warnings

import torch
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


# ============================================================
# Project Root
# ============================================================

PROJECT_ROOT = Path(r"D:\Gpt2_v01")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"MyGPT2 project root not found:\n{PROJECT_ROOT}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# ============================================================
# Project Paths
# ============================================================

MODEL_DIR = PROJECT_ROOT / "model"
TOKENIZER_DIR = PROJECT_ROOT / "artifacts" / "tokenizer"
CHECKPOINT_DIR = PROJECT_ROOT / "artifacts" / "checkpoints"

TOKENIZER_PATH = TOKENIZER_DIR / "tokenizer.json"


# ============================================================
# IMPORTANT:
# Your latest recorded checkpoint was:
#
#     step_00213000.pt
#
# ============================================================

CHECKPOINT_PATH = CHECKPOINT_DIR / "step_00213000.pt"


# ============================================================
# Training Log
# ============================================================

# We search for logs automatically rather than assuming
# a variable such as LOG_PATH already exists.

POSSIBLE_LOGS = [
    PROJECT_ROOT / "training.log",
    PROJECT_ROOT / "train.log",
    PROJECT_ROOT / "artifacts" / "training.log",
    PROJECT_ROOT / "artifacts" / "train.log",
    PROJECT_ROOT / "logs" / "training.log",
]

LOG_PATH = None

for candidate in POSSIBLE_LOGS:
    if candidate.exists():
        LOG_PATH = candidate
        break


# ============================================================
# Device
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


print("=" * 80)
print("MyGPT2 CHECKPOINT EVALUATION")
print("=" * 80)

print(f"Project Root : {PROJECT_ROOT}")
print(f"Checkpoint   : {CHECKPOINT_PATH}")
print(f"Tokenizer    : {TOKENIZER_PATH}")
print(f"Device       : {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")

print("=" * 80)

MyGPT2 CHECKPOINT EVALUATION
Project Root : D:\Gpt2_v01
Checkpoint   : D:\Gpt2_v01\artifacts\checkpoints\step_00213000.pt
Tokenizer    : D:\Gpt2_v01\artifacts\tokenizer\tokenizer.json
Device       : cuda
GPU          : NVIDIA GeForce RTX 5060 Ti


In [111]:
# ============================================================
# LOAD TRAINED CHECKPOINT WEIGHTS INTO MODEL
# ============================================================

print("=" * 80)
print("LOADING TRAINED MODEL WEIGHTS")
print("=" * 80)

# Load trained weights from the checkpoint state dict
load_result = model.load_state_dict(
    model_state_dict,
    strict=True,
)

# Move model to selected device
model = model.to(DEVICE)

# Switch to evaluation mode
model.eval()

print()
print(f"State source    : {model_state_key}")
print(f"Missing keys    : {len(load_result.missing_keys)}")
print(f"Unexpected keys : {len(load_result.unexpected_keys)}")
print(f"Model device    : {next(model.parameters()).device}")
print(f"Model dtype     : {next(model.parameters()).dtype}")

print()

if (
    len(load_result.missing_keys) == 0
    and len(load_result.unexpected_keys) == 0
):
    print("PASS: trained checkpoint weights loaded successfully.")
else:
    print("WARNING: checkpoint/model state mismatch detected.")

print("=" * 80)

LOADING TRAINED MODEL WEIGHTS

State source    : model_state_dict
Missing keys    : 0
Unexpected keys : 0
Model device    : cuda:0
Model dtype     : torch.float32

PASS: trained checkpoint weights loaded successfully.


In [112]:
# ============================================================
# Verify Files
# ============================================================

print("=" * 80)
print("VERIFYING FILES")
print("=" * 80)


if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root does not exist:\n{PROJECT_ROOT}"
    )

if not TOKENIZER_PATH.exists():
    raise FileNotFoundError(
        f"Tokenizer not found:\n{TOKENIZER_PATH}"
    )

if not CHECKPOINT_DIR.exists():
    raise FileNotFoundError(
        f"Checkpoint directory not found:\n{CHECKPOINT_DIR}"
    )

if not CHECKPOINT_PATH.exists():

    print()
    print("Requested checkpoint was not found.")
    print()
    print("Available checkpoints:")
    print("-" * 80)

    checkpoints = sorted(
        CHECKPOINT_DIR.glob("*.pt")
    )

    if not checkpoints:
        raise FileNotFoundError(
            "No .pt checkpoints were found."
        )

    for path in checkpoints:
        print(path.name)

    raise FileNotFoundError(
        f"\nCheckpoint not found:\n{CHECKPOINT_PATH}"
    )


print()
print("Project root     : OK")
print("Tokenizer        : OK")
print("Checkpoint dir   : OK")
print("Target checkpoint: OK")

print()
print(f"Checkpoint size  : "
      f"{CHECKPOINT_PATH.stat().st_size / (1024**3):.2f} GB")

print("=" * 80)

VERIFYING FILES

Project root     : OK
Tokenizer        : OK
Checkpoint dir   : OK
Target checkpoint: OK

Checkpoint size  : 1.23 GB


In [113]:
# ============================================================
# Load Project Components
# ============================================================

from model.config import GPTConfig
from model.model import MyGPTModel
from tokenizer.my_tokenizer import MyGPTTokenizer


print("Project classes imported successfully.")

print()
print("GPTConfig       :", GPTConfig)
print("MyGPTModel      :", MyGPTModel)
print("MyGPTTokenizer  :", MyGPTTokenizer)

Project classes imported successfully.

GPTConfig       : <class 'model.config.GPTConfig'>
MyGPTModel      : <class 'model.model.MyGPTModel'>
MyGPTTokenizer  : <class 'tokenizer.my_tokenizer.MyGPTTokenizer'>


In [114]:
# ============================================================
# Load Tokenizer
# ============================================================

print("=" * 80)
print("LOADING TOKENIZER")
print("=" * 80)

tokenizer = MyGPTTokenizer.load(
    TOKENIZER_PATH
)

print()
print("Tokenizer loaded successfully.")

print(
    f"Vocabulary Size : {tokenizer.vocabulary_size:,}"
)

print(
    f"Tokenizer Path  : {TOKENIZER_PATH}"
)

print("=" * 80)

LOADING TOKENIZER

Tokenizer loaded successfully.
Vocabulary Size : 32,000
Tokenizer Path  : D:\Gpt2_v01\artifacts\tokenizer\tokenizer.json


In [115]:
# ============================================================
# Load Checkpoint
# ============================================================

print("=" * 80)
print("LOADING CHECKPOINT")
print("=" * 80)

print()
print(f"Checkpoint:")
print(CHECKPOINT_PATH)

try:

    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=False,
    )

except TypeError:

    # Compatibility with older PyTorch versions

    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
    )


print()
print("Checkpoint loaded successfully.")

print()
print("Checkpoint type:")
print(type(checkpoint))


if isinstance(checkpoint, dict):

    print()
    print("Checkpoint keys:")

    for key in checkpoint.keys():
        print(f"  - {key}")

else:

    print(
        "\nWARNING: checkpoint is not a dictionary."
    )

print("=" * 80)

LOADING CHECKPOINT

Checkpoint:
D:\Gpt2_v01\artifacts\checkpoints\step_00213000.pt

Checkpoint loaded successfully.

Checkpoint type:
<class 'dict'>

Checkpoint keys:
  - checkpoint_version
  - created_at
  - epoch
  - global_step
  - best_loss
  - train_loss
  - val_loss
  - model_state_dict
  - optimizer_state_dict
  - scheduler_state_dict
  - random_states
  - config
  - extra
  - mygpt2_training_manifest
  - mygpt2_checkpoint_version
  - mygpt2_checkpoint_saved_at


In [116]:
# ============================================================
# Checkpoint Metadata
# ============================================================

print("=" * 80)
print("CHECKPOINT METADATA")
print("=" * 80)


def find_value(dictionary, names):

    if not isinstance(dictionary, dict):
        return None

    for name in names:

        if name in dictionary:
            return dictionary[name]

    return None


global_step = find_value(
    checkpoint,
    [
        "global_step",
        "step",
        "steps",
    ],
)

epoch = find_value(
    checkpoint,
    [
        "epoch",
        "current_epoch",
    ],
)

train_loss = find_value(
    checkpoint,
    [
        "train_loss",
        "current_train_loss",
        "loss",
    ],
)

val_loss = find_value(
    checkpoint,
    [
        "val_loss",
        "current_val_loss",
    ],
)


print()
print(f"Global Step : {global_step}")
print(f"Epoch       : {epoch}")
print(f"Train Loss  : {train_loss}")
print(f"Val Loss    : {val_loss}")


print()
print("Checkpoint metadata inspection complete.")

print("=" * 80)

CHECKPOINT METADATA

Global Step : 213000
Epoch       : 0
Train Loss  : 4.340525150299072
Val Loss    : None

Checkpoint metadata inspection complete.


In [117]:
# ============================================================
# Find Model State
# ============================================================

print("=" * 80)
print("LOCATING MODEL STATE")
print("=" * 80)


MODEL_STATE_KEYS = [
    "model_state_dict",
    "model_state",
    "state_dict",
    "model",
]


model_state_dict =None
model_state_key = None


if isinstance(checkpoint, dict):

    for key in MODEL_STATE_KEYS:

        value = checkpoint.get(key)

        if isinstance(value, dict):

            # Check whether it looks like a PyTorch state dict.

            tensor_values = [
                v
                for v in value.values()
                if torch.is_tensor(v)
            ]

            if tensor_values:

                model_state_dict = value
                model_state_key = key
                break


if model_state_dict is None:

    # Some checkpoints may themselves be state dictionaries.

    tensor_values = [
        v
        for v in checkpoint.values()
        if torch.is_tensor(v)
    ]

    if tensor_values:

        model_state_dict = checkpoint
        model_state_key = "<root>"


if model_state_dict is None:

    raise RuntimeError(
        "Could not locate model state dictionary "
        "inside the checkpoint."
    )


print()
print(f"Model state key : {model_state_key}")
print(
    f"Parameters      : "
    f"{len(model_state_dict):,}"
)

print("=" * 80)

LOCATING MODEL STATE

Model state key : model_state_dict
Parameters      : 149


In [118]:
# ============================================================
# Recover GPT Configuration
# ============================================================

print("=" * 80)
print("RECOVERING GPT CONFIGURATION")
print("=" * 80)


def extract_config_from_checkpoint(ckpt):

    if not isinstance(ckpt, dict):
        return None

    possible_keys = [
        "config",
        "model_config",
        "model_configuration",
    ]

    for key in possible_keys:

        value = ckpt.get(key)

        if value is None:
            continue

        if isinstance(value, dict):
            return value

        if hasattr(value, "__dict__"):
            return vars(value)

    return None


checkpoint_config = extract_config_from_checkpoint(
    checkpoint
)


if checkpoint_config is not None:

    print()
    print("Configuration found in checkpoint.")

    for key, value in checkpoint_config.items():
        print(f"{key:30}: {value}")

else:

    print()
    print(
        "No explicit configuration dictionary found."
    )

    print(
        "Using the known MyGPT2 final architecture."
    )

    checkpoint_config = {
        "vocab_size": 32000,
        "context_length": 512,
        "hidden_size": 768,
        "num_layers": 12,
        "num_heads": 12,
        "intermediate_size": 3072,
    }


print("=" * 80)

RECOVERING GPT CONFIGURATION

Configuration found in checkpoint.
vocab_size                    : 32000
max_position_embeddings       : 512
hidden_size                   : 768
num_layers                    : 12
num_attention_heads           : 12
intermediate_size             : 3072
dropout                       : 0.1
attention_dropout             : 0.1
embedding_dropout             : 0.1
layer_norm_epsilon            : 1e-05
initializer_range             : 0.02
batch_size                    : 8
learning_rate                 : 0.0003
weight_decay                  : 0.01
max_epochs                    : 10
gradient_clip                 : 1.0
temperature                   : 1.0
top_k                         : 50
top_p                         : 0.95
pad_token_id                  : 0
unk_token_id                  : 1
bos_token_id                  : 2
eos_token_id                  : 3
use_bias                      : True
device                        : cuda
seed                          : 42


In [119]:
# ============================================================
# CELL 9 — MODEL / CONFIGURATION INSPECTION
# ============================================================

print("=" * 75)
print("MODEL CONFIGURATION")
print("=" * 75)

# ------------------------------------------------------------
# Helper: safely retrieve configuration attributes
# ------------------------------------------------------------

def get_config_value(config, names, default=None):
    """
    Return the first existing configuration attribute.

    This makes the evaluation notebook compatible with
    different GPTConfig naming conventions.
    """
    for name in names:
        if hasattr(config, name):
            value = getattr(config, name)

            if value is not None:
                return value

    return default


# ------------------------------------------------------------
# Vocabulary size
# ------------------------------------------------------------

vocab_size = get_config_value(
    config,
    [
        "vocab_size",
        "vocabulary_size",
        "n_vocab",
    ],
    default=None,
)


# ------------------------------------------------------------
# Context / sequence length
# ------------------------------------------------------------

context_length = get_config_value(
    config,
    [
        "context_length",
        "seq_length",
        "sequence_length",
        "max_seq_len",
        "max_sequence_length",
        "block_size",
        "context_size",
        "n_positions",
    ],
    default=512,
)


# ------------------------------------------------------------
# Hidden size
# ------------------------------------------------------------

hidden_size = get_config_value(
    config,
    [
        "hidden_size",
        "d_model",
        "n_embd",
        "embedding_dim",
    ],
    default=None,
)


# ------------------------------------------------------------
# Number of transformer layers
# ------------------------------------------------------------

num_layers = get_config_value(
    config,
    [
        "num_layers",
        "n_layers",
        "num_hidden_layers",
        "layers",
    ],
    default=None,
)


# ------------------------------------------------------------
# Attention heads
# ------------------------------------------------------------

num_heads = get_config_value(
    config,
    [
        "num_heads",
        "n_heads",
        "num_attention_heads",
        "attention_heads",
    ],
    default=None,
)


# ------------------------------------------------------------
# Intermediate / FFN size
# ------------------------------------------------------------

intermediate_size = get_config_value(
    config,
    [
        "intermediate_size",
        "ffn_size",
        "hidden_dim",
        "n_inner",
    ],
    default=None,
)


# ------------------------------------------------------------
# Print configuration
# ------------------------------------------------------------

print()

print(
    f"Vocabulary Size      : "
    f"{vocab_size:,}"
    if isinstance(vocab_size, int)
    else
    f"Vocabulary Size      : {vocab_size}"
)

print(
    f"Context Length       : "
    f"{context_length:,}"
)

print(
    f"Hidden Size          : "
    f"{hidden_size:,}"
    if isinstance(hidden_size, int)
    else
    f"Hidden Size          : {hidden_size}"
)

print(
    f"Transformer Layers   : "
    f"{num_layers:,}"
    if isinstance(num_layers, int)
    else
    f"Transformer Layers   : {num_layers}"
)

print(
    f"Attention Heads      : "
    f"{num_heads:,}"
    if isinstance(num_heads, int)
    else
    f"Attention Heads      : {num_heads}"
)

print(
    f"Intermediate Size    : "
    f"{intermediate_size:,}"
    if isinstance(intermediate_size, int)
    else
    f"Intermediate Size    : {intermediate_size}"
)

# ------------------------------------------------------------
# Parameter count
# ------------------------------------------------------------

try:

    total_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    trainable_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    print(
        f"Total Parameters     : "
        f"{total_parameters:,}"
    )

    print(
        f"Total Parameters     : "
        f"{total_parameters / 1_000_000:.2f}M"
    )

    print(
        f"Trainable Parameters : "
        f"{trainable_parameters:,}"
    )

except Exception as exc:

    print(
        "Parameter count      : "
        f"Unable to calculate ({exc})"
    )


print()

print("=" * 75)
print("CONFIGURATION INSPECTION COMPLETED")
print("=" * 75)

MODEL CONFIGURATION

Vocabulary Size      : 32,000
Context Length       : 512
Hidden Size          : 768
Transformer Layers   : 12
Attention Heads      : 12
Intermediate Size    : 3,072
Total Parameters     : 110,025,216
Total Parameters     : 110.03M
Trainable Parameters : 110,025,216

CONFIGURATION INSPECTION COMPLETED


In [120]:
print("GPTConfig attributes:")
print("=" * 75)

for name in dir(config):
    if not name.startswith("_"):
        try:
            value = getattr(config, name)

            if not callable(value):
                print(f"{name:30} = {value}")

        except Exception:
            pass

GPTConfig attributes:
attention_dropout              = 0.1
batch_size                     = 8
bos_token_id                   = 2
device                         = cuda
dropout                        = 0.1
dtype                          = torch.float32
embedding_dropout              = 0.1
eos_token_id                   = 3
gradient_clip                  = 1.0
head_dim                       = 64
hidden_size                    = 768
initializer_range              = 0.02
intermediate_size              = 3072
layer_norm_epsilon             = 1e-05
learning_rate                  = 0.0003
max_epochs                     = 10
max_position_embeddings        = 512
model_size                     = GPT-2 Small
num_attention_heads            = 12
num_layers                     = 12
pad_token_id                   = 0
seed                           = 42
temperature                    = 1.0
top_k                          = 50
top_p                          = 0.95
total_attention_dimensions     = 768
unk

In [121]:
# ============================================================
# CELL 10 — CREATE MODEL AND LOAD CHECKPOINT
# ============================================================

import sys
import torch
from pathlib import Path

# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"D:\Gpt2_v01")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------

from model.model import MyGPTModel
from model.config import GPTConfig


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 75)
print("MODEL INITIALIZATION")
print("=" * 75)

print(
    f"Device : {DEVICE}"
)

if DEVICE.type == "cuda":

    print(
        f"GPU    : {torch.cuda.get_device_name(0)}"
    )


# ------------------------------------------------------------
# Check configuration object
# ------------------------------------------------------------

print()
print("Creating GPT model...")


# IMPORTANT:
# If your notebook already created `config` successfully,
# reuse it.
#
# Otherwise create a configuration with the values used
# during your actual training.

if "config" not in globals():

    print(
        "Configuration object not found."
    )

    config = GPTConfig(
        vocab_size=32000,
        context_length=512,
        hidden_size=768,
        num_layers=12,
        num_heads=12,
        intermediate_size=3072,
    )


# ------------------------------------------------------------
# Create model
# ------------------------------------------------------------

model = MyGPTModel(
    config
)

model = model.to(
    DEVICE
)

model.eval()


# ------------------------------------------------------------
# Parameter count
# ------------------------------------------------------------

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print()
print(
    f"Total Parameters     : "
    f"{total_parameters:,}"
)

print(
    f"Total Parameters     : "
    f"{total_parameters / 1_000_000:.2f}M"
)

print(
    f"Trainable Parameters : "
    f"{trainable_parameters:,}"
)


# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------

print()
print("=" * 75)
print("LOADING CHECKPOINT")
print("=" * 75)

print(
    f"Checkpoint : {CHECKPOINT_PATH}"
)

if not CHECKPOINT_PATH.exists():

    raise FileNotFoundError(
        f"Checkpoint does not exist:\n"
        f"{CHECKPOINT_PATH}"
    )


checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
)


# ------------------------------------------------------------
# Inspect checkpoint structure
# ------------------------------------------------------------

print()
print("Checkpoint loaded.")

if isinstance(checkpoint, dict):

    print()
    print("Checkpoint keys:")

    for key in checkpoint.keys():

        print(
            f"  - {key}"
        )


# ------------------------------------------------------------
# Locate model state
# ------------------------------------------------------------

if not isinstance(
    checkpoint,
    dict
):

    raise RuntimeError(
        "Checkpoint is not a dictionary."
    )


MODEL_STATE_KEYS = [
    "model_state_dict",
    "model",
    "state_dict",
]


model_state = None

for key in MODEL_STATE_KEYS:

    if key in checkpoint:

        model_state = checkpoint[key]

        print()
        print(
            f"Model state found under key: "
            f"'{key}'"
        )

        break


# ------------------------------------------------------------
# Some checkpoints may directly contain state_dict
# ------------------------------------------------------------

if model_state is None:

    # Detect whether this dictionary itself looks like
    # a PyTorch state_dict.

    if all(
        isinstance(k, str)
        for k in checkpoint.keys()
    ):

        tensor_values = [
            value
            for value in checkpoint.values()
            if torch.is_tensor(value)
        ]

        if len(tensor_values) > 0:

            model_state = checkpoint

            print()
            print(
                "Checkpoint itself appears to be "
                "a model state_dict."
            )


# ------------------------------------------------------------
# Validate model state
# ------------------------------------------------------------

if model_state is None:

    raise RuntimeError(
        "Could not find model weights in checkpoint.\n\n"
        "Expected one of:\n"
        "  model_state_dict\n"
        "  model\n"
        "  state_dict"
    )


# ------------------------------------------------------------
# Load model weights
# ------------------------------------------------------------

try:

    missing_keys, unexpected_keys = (
        model.load_state_dict(
            model_state,
            strict=False,
        )
    )

except RuntimeError as exc:

    raise RuntimeError(
        "Checkpoint model weights are incompatible "
        "with the current MyGPTModel configuration.\n\n"
        f"{exc}"
    )


# ------------------------------------------------------------
# Report loading result
# ------------------------------------------------------------

print()

if missing_keys:

    print(
        "WARNING — Missing model keys:"
    )

    for key in missing_keys[:20]:

        print(
            f"  {key}"
        )

    if len(missing_keys) > 20:

        print(
            f"  ... and "
            f"{len(missing_keys) - 20} more"
        )

else:

    print(
        "Missing Keys       : NONE"
    )


print()

if unexpected_keys:

    print(
        "WARNING — Unexpected model keys:"
    )

    for key in unexpected_keys[:20]:

        print(
            f"  {key}"
        )

    if len(unexpected_keys) > 20:

        print(
            f"  ... and "
            f"{len(unexpected_keys) - 20} more"
        )

else:

    print(
        "Unexpected Keys     : NONE"
    )


# ------------------------------------------------------------
# Finalize model
# ------------------------------------------------------------

model = model.to(
    DEVICE
)

model.eval()


print()
print("=" * 75)
print("MODEL LOADED SUCCESSFULLY")
print("=" * 75)

print(
    f"Checkpoint : {CHECKPOINT_PATH.name}"
)

print(
    f"Device     : {DEVICE}"
)

print(
    f"Parameters : "
    f"{total_parameters:,}"
)

print("=" * 75)

MODEL INITIALIZATION
Device : cuda
GPU    : NVIDIA GeForce RTX 5060 Ti

Creating GPT model...

Total Parameters     : 110,025,216
Total Parameters     : 110.03M
Trainable Parameters : 110,025,216

LOADING CHECKPOINT
Checkpoint : D:\Gpt2_v01\artifacts\checkpoints\step_00213000.pt


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray._reconstruct was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray._reconstruct])` or the `torch.serialization.safe_globals([numpy._core.multiarray._reconstruct])` context manager to allowlist this global if you trust this class/function.

Check the documentation of to

In [ ]:
# ============================================================
# Forward Pass Test
# ============================================================

print("=" * 80)
print("FORWARD PASS TEST")
print("=" * 80)


test_text = (
    "Artificial intelligence is transforming "
    "the way humans interact with computers."
)


encoded = tokenizer.encode(
    test_text
)


if not encoded:

    raise RuntimeError(
        "Tokenizer produced zero tokens."
    )


# Keep the test within model context length.
context_length = getattr(
    config,
    "context_length",
    getattr(config, "block_size", 512)
)

print(f"Context Length       : {context_length:,}")

encoded = encoded[:context_length]


input_ids = torch.tensor(
    [encoded],
    dtype=torch.long,
    device=DEVICE,
)


print()
print(f"Test text       : {test_text}")
print(f"Token count     : {input_ids.shape[1]}")
print(f"Input shape     : {tuple(input_ids.shape)}")


with torch.no_grad():

    output = model(
        input_ids=input_ids
    )


if isinstance(output, tuple):

    logits = output[0]

elif hasattr(output, "logits"):

    logits = output.logits

else:

    logits = output


print()
print(
    f"Logits shape    : "
    f"{tuple(logits.shape)}"
)

print(
    f"Expected vocab  : "
    f"{config.vocab_size:,}"
)


if logits.shape[-1] != config.vocab_size:

    raise RuntimeError(
        "Output vocabulary dimension does not "
        "match model vocabulary size."
    )


if not torch.isfinite(logits).all():

    raise RuntimeError(
        "Model produced NaN or Inf logits."
    )


print()
print("Forward pass : PASSED")

print("=" * 80)

FORWARD PASS TEST
Context Length       : 512

Test text       : Artificial intelligence is transforming the way humans interact with computers.
Token count     : 14
Input shape     : (1, 14)

Logits shape    : (1, 14, 32000)
Expected vocab  : 32,000

Forward pass : PASSED


In [ ]:
# ============================================================
# CELL 12 — CHECKPOINT INTEGRITY TEST
# ============================================================

print("=" * 75)
print("CHECKPOINT INTEGRITY TEST")
print("=" * 75)


# ------------------------------------------------------------
# Verify model exists
# ------------------------------------------------------------

if "model" not in globals():

    raise RuntimeError(
        "Model is not initialized.\n\n"
        "Run the model/checkpoint loading cell first."
    )


# ------------------------------------------------------------
# Check parameter tensors
# ------------------------------------------------------------

non_finite_parameters = []

total_parameter_tensors = 0

for name, parameter in model.named_parameters():

    total_parameter_tensors += 1

    if not torch.isfinite(
        parameter
    ).all():

        non_finite_parameters.append(
            name
        )


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print()
print(
    f"Parameter tensors checked : "
    f"{total_parameter_tensors:,}"
)

print(
    f"Non-finite tensors        : "
    f"{len(non_finite_parameters):,}"
)


if non_finite_parameters:

    print()
    print(
        "❌ CHECKPOINT INTEGRITY FAILED"
    )

    print()
    print(
        "Non-finite parameters:"
    )

    for name in non_finite_parameters:

        print(
            f"  - {name}"
        )

    raise RuntimeError(
        "Checkpoint contains NaN or Inf parameters."
    )


print()
print(
    "✅ All model parameters are finite."
)


# ------------------------------------------------------------
# Check parameter statistics
# ------------------------------------------------------------

all_finite = True

print()
print("Parameter statistics:")

for name, parameter in model.named_parameters():

    with torch.no_grad():

        minimum = parameter.min().item()
        maximum = parameter.max().item()
        mean = parameter.mean().item()
        std = parameter.std().item()

    print(
        f"{name:50} "
        f"min={minimum: .5e} "
        f"max={maximum: .5e} "
        f"mean={mean: .5e} "
        f"std={std: .5e}"
    )


# ------------------------------------------------------------
# Check checkpoint step
# ------------------------------------------------------------

checkpoint_step = None

if isinstance(checkpoint, dict):

    for key in [
        "global_step",
        "step",
        "training_step",
    ]:

        if key in checkpoint:

            checkpoint_step = checkpoint[key]

            break


print()

if checkpoint_step is not None:

    print(
        f"Checkpoint Global Step : "
        f"{checkpoint_step:,}"
    )

else:

    print(
        "Checkpoint Global Step : "
        "Not stored directly"
    )


# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print()
print("=" * 75)
print("CHECKPOINT INTEGRITY RESULT")
print("=" * 75)

print(
    "Model Parameters : ✅ FINITE"
)

print(
    "NaN Detection     : ✅ NONE"
)

print(
    "Inf Detection     : ✅ NONE"
)

print(
    "Checkpoint Load   : ✅ SUCCESS"
)

print(
    "Integrity Test    : ✅ PASSED"
)

print("=" * 75)

CHECKPOINT INTEGRITY TEST

Parameter tensors checked : 148
Non-finite tensors        : 0

✅ All model parameters are finite.

Parameter statistics:
embeddings.token_embeddings.weight                 min=-1.03956e-01 max= 1.11204e-01 mean=-1.44078e-07 std= 2.00035e-02
embeddings.position_embeddings.weight              min=-1.10339e-01 max= 8.77931e-02 mean= 4.17928e-05 std= 1.99926e-02
blocks.0.ln1.weight                                min= 1.00000e+00 max= 1.00000e+00 mean= 1.00000e+00 std= 0.00000e+00
blocks.0.ln1.bias                                  min= 0.00000e+00 max= 0.00000e+00 mean= 0.00000e+00 std= 0.00000e+00
blocks.0.attention.qkv_proj.weight                 min=-9.67722e-02 max= 1.05082e-01 mean=-7.37524e-06 std= 1.99981e-02
blocks.0.attention.qkv_proj.bias                   min= 0.00000e+00 max= 0.00000e+00 mean= 0.00000e+00 std= 0.00000e+00
blocks.0.attention.out_proj.weight                 min=-1.86957e-02 max= 1.92946e-02 mean=-1.13767e-05 std= 4.07830e-03
blocks.0.att

In [ ]:
import math
import torch


@torch.no_grad()
def calculate_loss_and_perplexity(
    model,
    tokenizer,
    texts,
    sequence_length=512,
    device=None,
):
    """
    Calculate average next-token cross-entropy loss
    and perplexity over evaluation texts.
    """

    model.eval()

    # Resolve device
    if device is None:
        device = next(model.parameters()).device

    # Make sure sequence length is an integer
    sequence_length = int(sequence_length)

    total_loss = 0.0
    total_tokens = 0

    for text in texts:

        if not text or not text.strip():
            continue

        # ----------------------------------------------------
        # Tokenize
        # ----------------------------------------------------

        encoded = tokenizer.encode(text)

        # Handle tokenizer returning a dictionary
        if isinstance(encoded, dict):
            encoded = encoded["input_ids"]

        # Convert to tensor
        if not isinstance(encoded, torch.Tensor):
            encoded = torch.tensor(
                encoded,
                dtype=torch.long
            )

        encoded = encoded.to(device)

        # Need at least 2 tokens
        if encoded.numel() < 2:
            continue

        # ----------------------------------------------------
        # Evaluate sequence in chunks
        # ----------------------------------------------------

        for start in range(
            0,
            encoded.numel() - 1,
            sequence_length
        ):

            chunk = encoded[
                start:start + sequence_length + 1
            ]

            if chunk.numel() < 2:
                continue

            # Input tokens
            input_ids = chunk[:-1].unsqueeze(0)

            # Next-token labels
            labels = chunk[1:].unsqueeze(0)

            # ------------------------------------------------
            # Forward pass
            # ------------------------------------------------

            output = model(
                input_ids=input_ids,
                labels=labels
            )

            # ------------------------------------------------
            # Extract loss
            # ------------------------------------------------

            if hasattr(output, "loss") and output.loss is not None:

                loss = output.loss

            elif isinstance(output, dict) and "loss" in output:

                loss = output["loss"]

            elif isinstance(output, (tuple, list)):

                loss = output[-1]

            else:

                raise RuntimeError(
                    "Could not extract loss from model output."
                )

            # ------------------------------------------------
            # Accumulate token-weighted loss
            # ------------------------------------------------

            num_tokens = labels.numel()

            total_loss += (
                loss.item() * num_tokens
            )

            total_tokens += num_tokens

    # --------------------------------------------------------
    # Safety check
    # --------------------------------------------------------

    if total_tokens == 0:

        raise ValueError(
            "No valid tokens were evaluated."
        )

    # --------------------------------------------------------
    # Average loss
    # --------------------------------------------------------

    average_loss = (
        total_loss / total_tokens
    )

    # --------------------------------------------------------
    # Perplexity
    # --------------------------------------------------------

    perplexity = math.exp(average_loss)

    return average_loss, perplexity

In [ ]:
# ============================================================
# Quick Held-Out Evaluation
# ============================================================

evaluation_texts = [

    """
    Machine learning is a field of artificial intelligence
    that allows computer systems to learn patterns from data
    and use those patterns to make predictions.
    """,

    """
    The transformer architecture uses attention mechanisms
    to process relationships between tokens in a sequence.
    """,

    """
    A language model learns statistical relationships between
    tokens and can generate new text based on a sequence of
    previously observed tokens.
    """,

    """
    Python is a popular programming language used for machine
    learning, scientific computing, automation, and web
    development.
    """,

    """
    Neural networks contain layers of mathematical operations
    that transform input representations into useful outputs.
    """,

]


# ============================================================
# Use the ACTUAL GPTConfig field
# ============================================================

sequence_length = config.max_position_embeddings

print(f"Evaluation Sequence Length : {sequence_length}")


# ============================================================
# Calculate loss and perplexity
# ============================================================

eval_loss, eval_perplexity = calculate_loss_and_perplexity(
    model=model,
    tokenizer=tokenizer,
    texts=evaluation_texts,
    sequence_length=sequence_length,
    device=DEVICE,
)


# ============================================================
# Results
# ============================================================

print("=" * 80)
print("EVALUATION RESULT")
print("=" * 80)

print()

print(
    f"Evaluation Loss       : "
    f"{eval_loss:.6f}"
)

print(
    f"Evaluation Perplexity : "
    f"{eval_perplexity:.4f}"
)

print()

print("NOTE:")

print(
    "This is a sanity-check evaluation, not a "
    "scientifically valid benchmark."
)

print("=" * 80)

Evaluation Sequence Length : 512
EVALUATION RESULT

Evaluation Loss       : 10.394073
Evaluation Perplexity : 32665.4559

NOTE:
This is a sanity-check evaluation, not a scientifically valid benchmark.


In [ ]:
print(vars(config))

print("GPTConfig attributes:")
print("=" * 80)

for key, value in vars(config).items():
    print(f"{key:30} : {value}")

{'vocab_size': 32000, 'max_position_embeddings': 512, 'hidden_size': 768, 'num_layers': 12, 'num_attention_heads': 12, 'intermediate_size': 3072, 'dropout': 0.1, 'attention_dropout': 0.1, 'embedding_dropout': 0.1, 'layer_norm_epsilon': 1e-05, 'initializer_range': 0.02, 'batch_size': 8, 'learning_rate': 0.0003, 'weight_decay': 0.01, 'max_epochs': 10, 'gradient_clip': 1.0, 'temperature': 1.0, 'top_k': 50, 'top_p': 0.95, 'pad_token_id': 0, 'unk_token_id': 1, 'bos_token_id': 2, 'eos_token_id': 3, 'use_bias': True, 'device': 'cuda', 'seed': 42}
GPTConfig attributes:
vocab_size                     : 32000
max_position_embeddings        : 512
hidden_size                    : 768
num_layers                     : 12
num_attention_heads            : 12
intermediate_size              : 3072
dropout                        : 0.1
attention_dropout              : 0.1
embedding_dropout              : 0.1
layer_norm_epsilon             : 1e-05
initializer_range              : 0.02
batch_size           

In [ ]:
# ============================================================
# CELL 14 — TRAINING LOG ANALYSIS
# ============================================================

import re
from pathlib import Path

print("=" * 80)
print("TRAINING LOG ANALYSIS")
print("=" * 80)

PROJECT_ROOT = Path(r"D:\Gpt2_v01")
LOG_DIR = PROJECT_ROOT / "logs"

POSSIBLE_LOGS = [
    LOG_DIR / "training.log",
    LOG_DIR / "train.log",
    PROJECT_ROOT / "training.log",
    PROJECT_ROOT / "train.log",
]

# Always create these variables.
# This prevents later cells from failing if no log exists.
steps = []
losses = []
learning_rates = []

LOG_PATH = None

for path in POSSIBLE_LOGS:
    if path.exists():
        LOG_PATH = path
        break


if LOG_PATH is None:

    print()
    print("No dedicated training log found.")
    print("Training-history analysis will be skipped.")
    print()

    print("Searched locations:")

    for path in POSSIBLE_LOGS:
        print(f"  {path}")

    print()

    dataset_log = LOG_DIR / "dataset_manager.log"

    if dataset_log.exists():
        print(
            "dataset_manager.log exists, but it is not "
            "a GPT training-step log."
        )

else:

    print()
    print(f"Training log: {LOG_PATH}")

    log_text = LOG_PATH.read_text(
        encoding="utf-8",
        errors="ignore",
    )

    pattern = re.compile(
        r"Step\s+(\d+)\s*\|\s*"
        r"Loss\s+([0-9.eE+-]+)\s*\|\s*"
        r"LR\s+([0-9.eE+-]+)"
    )

    matches = pattern.findall(log_text)

    for step, loss, lr in matches:
        steps.append(int(step))
        losses.append(float(loss))
        learning_rates.append(float(lr))

    print()
    print(f"Logged steps : {len(steps):,}")

    if steps:
        print(f"First step   : {steps[0]:,}")
        print(f"Last step    : {steps[-1]:,}")
        print(f"First loss   : {losses[0]:.6f}")
        print(f"Last loss    : {losses[-1]:.6f}")
        print(f"First LR     : {learning_rates[0]:.8f}")
        print(f"Last LR      : {learning_rates[-1]:.8f}")

    else:
        print("No valid training-step records found.")

print("=" * 80)

TRAINING LOG ANALYSIS

No dedicated training log found.
Training-history analysis will be skipped.

Searched locations:
  D:\Gpt2_v01\logs\training.log
  D:\Gpt2_v01\logs\train.log
  D:\Gpt2_v01\training.log
  D:\Gpt2_v01\train.log

dataset_manager.log exists, but it is not a GPT training-step log.


In [ ]:
# ============================================================
# CELL 15 — TRAINING LOSS CURVE
# ============================================================

import matplotlib.pyplot as plt

print("=" * 80)
print("TRAINING LOSS CURVE")
print("=" * 80)

if len(steps) > 0 and len(losses) > 0:

    plt.figure(figsize=(12, 5))

    plt.plot(
        steps,
        losses,
        linewidth=1.0,
    )

    plt.xlabel("Training Step")
    plt.ylabel("Loss")
    plt.title("GPT Training Loss")
    plt.grid(alpha=0.3)

    plt.show()

else:

    print()
    print(
        "Training loss plot skipped because "
        "no training log is available."
    )

print("=" * 80)

TRAINING LOSS CURVE

Training loss plot skipped because no training log is available.


In [ ]:
# ============================================================
# CELL 16 — LEARNING RATE CURVE
# ============================================================

print("=" * 80)
print("LEARNING RATE CURVE")
print("=" * 80)

if len(steps) > 0 and len(learning_rates) > 0:

    plt.figure(figsize=(12, 5))

    plt.plot(
        steps,
        learning_rates,
        linewidth=1.0,
    )

    plt.xlabel("Training Step")
    plt.ylabel("Learning Rate")
    plt.title("Learning Rate Schedule")
    plt.grid(alpha=0.3)

    plt.show()

else:

    print()
    print(
        "Learning-rate plot skipped because "
        "no training log is available."
    )

print("=" * 80)

LEARNING RATE CURVE

Learning-rate plot skipped because no training log is available.


In [ ]:
# ============================================================
# CELL 17 — MODEL CONFIGURATION SUMMARY
# ============================================================

print("=" * 80)
print("MODEL CONFIGURATION")
print("=" * 80)

config_fields = [
    "vocab_size",
    "max_position_embeddings",
    "hidden_size",
    "num_layers",
    "num_attention_heads",
    "intermediate_size",
    "dropout",
    "attention_dropout",
    "embedding_dropout",
]

for field in config_fields:

    value = getattr(
        config,
        field,
        "NOT AVAILABLE"
    )

    print(
        f"{field:<30}: {value}"
    )

print("=" * 80)

MODEL CONFIGURATION
vocab_size                    : 32000
max_position_embeddings       : 512
hidden_size                   : 768
num_layers                    : 12
num_attention_heads           : 12
intermediate_size             : 3072
dropout                       : 0.1
attention_dropout             : 0.1
embedding_dropout             : 0.1


In [ ]:
# ============================================================
# FINAL TRAINING STATISTICS
# ============================================================

print("=" * 80)
print("FINAL TRAINING STATISTICS")
print("=" * 80)


if LOG_PATH is not None and steps:

    losses_np = np.asarray(
        losses,
        dtype=np.float64,
    )

    print()

    print(
        f"Logged Steps        : "
        f"{len(steps):,}"
    )

    print(
        f"First Step          : "
        f"{steps[0]:,}"
    )

    print(
        f"Last Step           : "
        f"{steps[-1]:,}"
    )

    print(
        f"Minimum Loss        : "
        f"{losses_np.min():.6f}"
    )

    print(
        f"Maximum Loss        : "
        f"{losses_np.max():.6f}"
    )

    print(
        f"Mean Loss           : "
        f"{losses_np.mean():.6f}"
    )

    print(
        f"Last Loss           : "
        f"{losses_np[-1]:.6f}"
    )

    print(
        f"Last Learning Rate  : "
        f"{learning_rates[-1]:.10f}"
    )


    # --------------------------------------------------------
    # Compare beginning and end
    # --------------------------------------------------------

    n = min(
        1000,
        len(losses_np) // 2,
    )


    if n > 0:

        beginning_mean = (
            losses_np[:n].mean()
        )

        ending_mean = (
            losses_np[-n:].mean()
        )

        print()

        print(
            f"First {n:,} mean loss : "
            f"{beginning_mean:.6f}"
        )

        print(
            f"Last {n:,} mean loss  : "
            f"{ending_mean:.6f}"
        )

        improvement = (
            (
                beginning_mean
                - ending_mean
            )
            / beginning_mean
            * 100
        )

        print(
            f"Loss improvement      : "
            f"{improvement:.2f}%"
        )

else:

    print()
    print(
        "Training log unavailable."
    )

print("=" * 80)

FINAL TRAINING STATISTICS

Training log unavailable.


In [ ]:
# ============================================================
# CELL 18 — PARAMETER COUNT
# ============================================================

print("=" * 80)
print("MODEL PARAMETER COUNT")
print("=" * 80)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

non_trainable_parameters = (
    total_parameters
    - trainable_parameters
)

print()
print(
    f"Total parameters      : "
    f"{total_parameters:,}"
)

print(
    f"Trainable parameters  : "
    f"{trainable_parameters:,}"
)

print(
    f"Frozen parameters     : "
    f"{non_trainable_parameters:,}"
)

print()

print(
    f"Approx model size FP32: "
    f"{total_parameters * 4 / (1024 ** 2):.2f} MB"
)

print("=" * 80)

MODEL PARAMETER COUNT

Total parameters      : 110,025,216
Trainable parameters  : 110,025,216
Frozen parameters     : 0

Approx model size FP32: 419.71 MB


In [ ]:
# ============================================================
# TRAINING COMPLETION CHECK
# ============================================================

EXPECTED_TOTAL_STEPS = 213_751


print("=" * 80)
print("TRAINING COMPLETION CHECK")
print("=" * 80)


checkpoint_step = global_step


print()

print(
    f"Expected Total Steps : "
    f"{EXPECTED_TOTAL_STEPS:,}"
)

print(
    f"Checkpoint Step      : "
    f"{checkpoint_step:,}"
)


if checkpoint_step is not None:

    progress = (
        checkpoint_step
        / EXPECTED_TOTAL_STEPS
        * 100
    )

    print(
        f"Training Progress    : "
        f"{progress:.2f}%"
    )


    if checkpoint_step >= EXPECTED_TOTAL_STEPS:

        print()
        print(
            "STATUS: COMPLETE"
        )

    else:

        remaining = (
            EXPECTED_TOTAL_STEPS
            - checkpoint_step
        )

        print()
        print(
            f"Remaining Steps      : "
            f"{remaining:,}"
        )

else:

    print()
    print(
        "Could not determine checkpoint step."
    )

print("=" * 80)

TRAINING COMPLETION CHECK

Expected Total Steps : 213,751
Checkpoint Step      : 213,000
Training Progress    : 99.65%

Remaining Steps      : 751


In [ ]:
# ============================================================
# TEXT GENERATION
# ============================================================

def generate_text(
    model,
    tokenizer,
    prompt,
    max_new_tokens=100,
    temperature=0.8,
    top_k=50,
    device=DEVICE,
):

    model.eval()

    token_ids = tokenizer.encode(
        prompt
    )

    if not token_ids:

        raise ValueError(
            "Prompt produced no tokens."
        )


    input_ids = torch.tensor(
        [token_ids],
        dtype=torch.long,
        device=device,
    )


    with torch.no_grad():

        for _ in range(
            max_new_tokens
        ):

            # Keep only context window.

            input_context = input_ids[
                :,
                -config.context_length:
            ]


            output = model(
                input_context
            )


            if isinstance(output, tuple):

                logits = output[0]

            elif hasattr(output, "logits"):

                logits = output.logits

            else:

                logits = output


            next_token_logits = (
                logits[:, -1, :]
            )


            # Temperature

            next_token_logits = (
                next_token_logits
                / max(temperature, 1e-6)
            )


            # Top-k

            if top_k is not None:

                k = min(
                    top_k,
                    next_token_logits.size(-1),
                )

                values, indices = torch.topk(
                    next_token_logits,
                    k=k,
                    dim=-1,
                )

                filtered = torch.full_like(
                    next_token_logits,
                    float("-inf"),
                )

                filtered.scatter_(
                    1,
                    indices,
                    values,
                )

                next_token_logits = filtered


            probabilities = torch.softmax(
                next_token_logits,
                dim=-1,
            )


            next_token = torch.multinomial(
                probabilities,
                num_samples=1,
            )


            input_ids = torch.cat(
                [
                    input_ids,
                    next_token,
                ],
                dim=1,
            )


    generated_ids = (
        input_ids[0]
        .detach()
        .cpu()
        .tolist()
    )


    # Use tokenizer decode method.

    try:

        return tokenizer.decode(
            generated_ids
        )

    except Exception:

        # Fallback if tokenizer has a different API.

        return str(generated_ids)

In [ ]:
# ============================================================
# GENERATION TESTS
# ============================================================

prompts = [
    "Artificial intelligence is",
    "The future of machine learning",
    "Once upon a time",
    "A computer scientist was",
    "Deep learning models",
]


print("=" * 80)
print("GENERATION EVALUATION")
print("=" * 80)


for index, prompt in enumerate(
    prompts,
    start=1,
):

    print()
    print("-" * 80)

    print(
        f"TEST {index}"
    )

    print(
        f"Prompt: {prompt}"
    )

    try:

        generated = generate_text(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            max_new_tokens=100,
            temperature=0.8,
            top_k=50,
            device=DEVICE,
        )

        print()
        print(
            generated
        )

    except Exception as exc:

        print()
        print(
            "Generation failed:"
        )

        print(
            repr(exc)
        )


print()
print("=" * 80)

GENERATION EVALUATION

--------------------------------------------------------------------------------
TEST 1
Prompt: Artificial intelligence is

Generation failed:
AttributeError("'GPTConfig' object has no attribute 'context_length'")

--------------------------------------------------------------------------------
TEST 2
Prompt: The future of machine learning

Generation failed:
AttributeError("'GPTConfig' object has no attribute 'context_length'")

--------------------------------------------------------------------------------
TEST 3
Prompt: Once upon a time

Generation failed:
AttributeError("'GPTConfig' object has no attribute 'context_length'")

--------------------------------------------------------------------------------
TEST 4
Prompt: A computer scientist was

Generation failed:
AttributeError("'GPTConfig' object has no attribute 'context_length'")

--------------------------------------------------------------------------------
TEST 5
Prompt: Deep learning models

Generat

In [ ]:
# ============================================================
# REPETITION TEST
# ============================================================

print("=" * 80)
print("REPETITION / DEGENERATION TEST")
print("=" * 80)


prompt = (
    "The development of artificial intelligence "
    "has changed"
)


generated = generate_text(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    max_new_tokens=200,
    temperature=0.8,
    top_k=50,
)


print()
print(generated)


# ------------------------------------------------------------
# Simple repetition statistics
# ------------------------------------------------------------

words = generated.split()

if words:

    unique_words = len(
        set(
            word.lower()
            for word in words
        )
    )

    total_words = len(words)

    diversity = (
        unique_words
        / total_words
        * 100
    )

    print()
    print("-" * 80)

    print(
        f"Total words       : "
        f"{total_words}"
    )

    print(
        f"Unique words      : "
        f"{unique_words}"
    )

    print(
        f"Lexical diversity : "
        f"{diversity:.2f}%"
    )

print("=" * 80)

REPETITION / DEGENERATION TEST


NameError: name 'model' is not defined

In [ ]:
# ============================================================
# FINAL EVALUATION SUMMARY
# ============================================================

print()
print("=" * 80)
print("MyGPT2 FINAL CHECKPOINT EVALUATION SUMMARY")
print("=" * 80)

print()

print(
    f"Project Root          : {PROJECT_ROOT}"
)

print(
    f"Checkpoint             : {CHECKPOINT_PATH.name}"
)

print(
    f"Checkpoint Step        : {global_step}"
)

print(
    f"Vocabulary Size        : {config.vocab_size:,}"
)

print(
    f"Context Length         : {config.context_length}"
)

print(
    f"Transformer Layers     : {config.num_layers}"
)

print(
    f"Attention Heads        : {config.num_heads}"
)

print(
    f"Hidden Size            : {config.hidden_size}"
)

print(
    f"Model Parameters       : "
    f"{total_parameters / 1e6:.2f}M"
)

print(
    f"Evaluation Loss        : "
    f"{eval_loss:.6f}"
)

print(
    f"Evaluation Perplexity  : "
    f"{eval_perplexity:.4f}"
)

print()

print(
    "Checkpoint Integrity   : PASSED"
)

print(
    "Forward Pass           : PASSED"
)

print(
    "Generation Test        : COMPLETED"
)

print()

print("=" * 80)
print("EVALUATION COMPLETE")
print("=" * 80)


MyGPT2 FINAL CHECKPOINT EVALUATION SUMMARY

Project Root          : D:\Gpt2_v01
Checkpoint             : step_00213000.pt
Checkpoint Step        : 213000
Vocabulary Size        : 32,000


AttributeError: 'GPTConfig' object has no attribute 'context_length'

In [ ]:
# ============================================================
# CELL 19 — DEVICE / MODEL SANITY CHECK
# ============================================================

import torch

print("=" * 80)
print("DEVICE AND MODEL CHECK")
print("=" * 80)

model_device = next(
    model.parameters()
).device

model_dtype = next(
    model.parameters()
).dtype

print()
print(
    f"Requested device : {DEVICE}"
)

print(
    f"Model device     : {model_device}"
)

print(
    f"Model dtype      : {model_dtype}"
)

print(
    f"CUDA available   : {torch.cuda.is_available()}"
)

if torch.cuda.is_available():

    print(
        f"CUDA device      : "
        f"{torch.cuda.get_device_name(0)}"
    )

print("=" * 80)

DEVICE AND MODEL CHECK

Requested device : cuda
Model device     : cuda:0
Model dtype      : torch.float32
CUDA available   : True
CUDA device      : NVIDIA GeForce RTX 5060 Ti


In [ ]:
# ============================================================
# CELL 20 — TOKENIZER SANITY CHECK
# ============================================================

print("=" * 80)
print("TOKENIZER SANITY CHECK")
print("=" * 80)

sample_text = (
    "Artificial intelligence can learn patterns from data."
)

encoded_sample = tokenizer.encode(
    sample_text
)

# Handle possible dictionary tokenizer output
if isinstance(encoded_sample, dict):
    encoded_sample = encoded_sample.get(
        "input_ids",
        encoded_sample
    )

print()
print(
    f"Input text : {sample_text}"
)

print(
    f"Encoded    : {encoded_sample}"
)

try:

    decoded_sample = tokenizer.decode(
        encoded_sample
    )

    print(
        f"Decoded    : {decoded_sample}"
    )

except Exception as error:

    print()
    print(
        "Tokenizer decode check could not be completed."
    )

    print(
        f"Reason: {error}"
    )

print("=" * 80)

TOKENIZER SANITY CHECK

Input text : Artificial intelligence can learn patterns from data.
Encoded    : [2, 4330, 30275, 4871, 438, 2049, 7527, 400, 1698, 17, 3]
Decoded    :  Artificial intelligence can learn patterns from data.


In [ ]:
# ============================================================
# CELL 21 — VOCABULARY CHECK
# ============================================================

print("=" * 80)
print("VOCABULARY CHECK")
print("=" * 80)

print()
print(
    f"Config vocabulary size : "
    f"{config.vocab_size:,}"
)

try:

    tokenizer_size = len(tokenizer)

    print(
        f"Tokenizer size         : "
        f"{tokenizer_size:,}"
    )

    if tokenizer_size == config.vocab_size:

        print()
        print(
            "Vocabulary sizes match."
        )

    else:

        print()
        print(
            "WARNING: tokenizer vocabulary size and "
            "model vocabulary size differ."
        )

except Exception:

    print()
    print(
        "Tokenizer does not expose len(tokenizer)."
    )

print("=" * 80)

VOCABULARY CHECK

Config vocabulary size : 32,000

Tokenizer does not expose len(tokenizer).


In [ ]:
# ============================================================
# CELL 22 — FORWARD PASS SANITY CHECK
# ============================================================

import torch

print("=" * 80)
print("FORWARD PASS SANITY CHECK")
print("=" * 80)

model.eval()

test_text = (
    "The future of artificial intelligence"
)

encoded = tokenizer.encode(
    test_text
)

if isinstance(encoded, dict):
    encoded = encoded["input_ids"]

if not isinstance(encoded, torch.Tensor):

    encoded = torch.tensor(
        encoded,
        dtype=torch.long,
    )

# Flatten in case tokenizer returns [1, seq]
encoded = encoded.reshape(-1)

# Respect actual model context limit
encoded = encoded[
    :config.max_position_embeddings
]

input_ids = encoded.unsqueeze(0).to(
    next(model.parameters()).device
)

print()
print(
    f"Input shape : {tuple(input_ids.shape)}"
)

try:

    with torch.no_grad():

        output = model(
            input_ids=input_ids
        )

    print(
        "Forward pass completed successfully."
    )

    # Try to identify logits
    if hasattr(output, "logits"):

        logits = output.logits

    elif isinstance(output, dict) and "logits" in output:

        logits = output["logits"]

    elif isinstance(output, torch.Tensor):

        logits = output

    elif isinstance(output, (tuple, list)):

        logits = output[0]

    else:

        logits = None


    if logits is not None:

        print(
            f"Logits shape: {tuple(logits.shape)}"
        )

        print(
            f"Logits dtype: {logits.dtype}"
        )

    else:

        print(
            "Forward pass succeeded, but logits "
            "could not be automatically identified."
        )


except Exception as error:

    print()
    print(
        "Forward pass failed."
    )

    print(
        f"Error type : {type(error).__name__}"
    )

    print(
        f"Error      : {error}"
    )

print("=" * 80)

FORWARD PASS SANITY CHECK

Input shape : (1, 7)
Forward pass completed successfully.
Logits shape: (1, 7, 32000)
Logits dtype: torch.float32


In [ ]:
# ============================================================
# CELL 23 — NUMERICAL STABILITY CHECK
# ============================================================

print("=" * 80)
print("NUMERICAL STABILITY CHECK")
print("=" * 80)

if "logits" in globals() and logits is not None:

    has_nan = torch.isnan(
        logits
    ).any().item()

    has_inf = torch.isinf(
        logits
    ).any().item()

    print()
    print(
        f"Contains NaN : {has_nan}"
    )

    print(
        f"Contains Inf : {has_inf}"
    )

    if not has_nan and not has_inf:

        print()
        print(
            "Numerical output looks healthy."
        )

    else:

        print()
        print(
            "WARNING: invalid numerical values "
            "were detected."
        )

else:

    print()
    print(
        "No logits are available from the previous "
        "cell, so this check was skipped."
    )

print("=" * 80)

NUMERICAL STABILITY CHECK

Contains NaN : False
Contains Inf : False

Numerical output looks healthy.


In [ ]:
# ============================================================
# CELL 24 — HELD-OUT EVALUATION SUMMARY
# ============================================================

print("=" * 80)
print("HELD-OUT EVALUATION SUMMARY")
print("=" * 80)

print()

if (
    "eval_loss" in globals()
    and
    "eval_perplexity" in globals()
):

    print(
        f"Evaluation Loss       : "
        f"{eval_loss:.6f}"
    )

    print(
        f"Evaluation Perplexity : "
        f"{eval_perplexity:.4f}"
    )

else:

    print(
        "Held-out evaluation results are unavailable."
    )

    print(
        "Run the Quick Held-Out Evaluation cell first."
    )

print()

print(
    "This is a sanity-check evaluation and should "
    "not be treated as a standard benchmark result."
)

print("=" * 80)

HELD-OUT EVALUATION SUMMARY

Evaluation Loss       : 10.394073
Evaluation Perplexity : 32665.4559

This is a sanity-check evaluation and should not be treated as a standard benchmark result.


In [ ]:
# ============================================================
# TEXT GENERATION EFFICIENCY TEST
# ============================================================

import time
import torch

print("=" * 80)
print("TEXT GENERATION EFFICIENCY TEST")
print("=" * 80)

prompt = (
    "Artificial intelligence is changing the way humans"
)

MAX_NEW_TOKENS = 100

model.eval()

# ------------------------------------------------------------
# Tokenize prompt
# ------------------------------------------------------------

encoded = tokenizer.encode(prompt)

if isinstance(encoded, dict):
    encoded = encoded["input_ids"]

if not isinstance(encoded, torch.Tensor):
    encoded = torch.tensor(
        encoded,
        dtype=torch.long,
    )

encoded = encoded.reshape(-1)

input_ids = encoded.unsqueeze(0).to(
    next(model.parameters()).device
)

prompt_tokens = input_ids.shape[1]

print()
print(f"Prompt              : {prompt}")
print(f"Prompt tokens       : {prompt_tokens}")
print(f"Requested new tokens: {MAX_NEW_TOKENS}")

# ------------------------------------------------------------
# CUDA preparation
# ------------------------------------------------------------

if torch.cuda.is_available():

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()

    # Important for accurate timing
    torch.cuda.synchronize()

# ------------------------------------------------------------
# Generation timing
# ------------------------------------------------------------

start_time = time.perf_counter()

with torch.no_grad():

    generated = model.generate(
        input_ids=input_ids,
        max_new_tokens=MAX_NEW_TOKENS,
    )

if torch.cuda.is_available():
    torch.cuda.synchronize()

end_time = time.perf_counter()

# ------------------------------------------------------------
# Measurements
# ------------------------------------------------------------

elapsed_time = end_time - start_time

generated = generated.reshape(-1)

total_tokens = generated.numel()

new_tokens = max(
    0,
    total_tokens - prompt_tokens
)

tokens_per_second = (
    new_tokens / elapsed_time
    if elapsed_time > 0
    else 0.0
)

ms_per_token = (
    (elapsed_time / new_tokens) * 1000
    if new_tokens > 0
    else 0.0
)

# ------------------------------------------------------------
# Decode
# ------------------------------------------------------------

generated_text = tokenizer.decode(
    generated.tolist()
)

# ------------------------------------------------------------
# GPU memory
# ------------------------------------------------------------

if torch.cuda.is_available():

    peak_memory_mb = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 2)
    )

else:

    peak_memory_mb = 0.0

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print()
print("=" * 80)
print("GENERATION RESULT")
print("=" * 80)

print()
print(generated_text)

print()
print("=" * 80)
print("GENERATION PERFORMANCE")
print("=" * 80)

print()
print(
    f"Generated tokens      : "
    f"{new_tokens}"
)

print(
    f"Generation time       : "
    f"{elapsed_time:.4f} sec"
)

print(
    f"Tokens / second       : "
    f"{tokens_per_second:.2f}"
)

print(
    f"Milliseconds / token  : "
    f"{ms_per_token:.2f} ms"
)

if torch.cuda.is_available():

    print(
        f"Peak GPU memory       : "
        f"{peak_memory_mb:.2f} MB"
    )

print("=" * 80)

TEXT GENERATION EFFICIENCY TEST

Prompt              : Artificial intelligence is changing the way humans
Prompt tokens       : 10
Requested new tokens: 100

GENERATION RESULT

 Artificial intelligence is changing the way humansResearchiling EqualityND concentrationUs prowakis Wesley Teamsthem 10 Word dominated suspiciousDER corridor hospitalsysis escapingppe Symphonyisb Im documentation Lap infants exha States contracting Percy 1935 openings loyalty stepsrench LegendCoverware specify @.@ restoringGrace cray countiesalking Defence identifiedFE reiteratedutaning letting confisc oral served joyful rout embargo menstr plantnett wonderedHarryPropYet guessing warns Wizardsfiredemail cypin tale shortcomingsatorerence oil fisherman Forbes rook True suittrans Filip crocod Ball Interestinglyitbartvia power combining Lizzie tweets mathematic RhrantApple parameters

GENERATION PERFORMANCE

Generated tokens      : 100
Generation time       : 0.9170 sec
Tokens / second       : 109.05
Milliseconds 

In [ ]:
# ============================================================
# CELL 25 — FINAL CHECKPOINT EVALUATION SUMMARY
# ============================================================

print("=" * 80)
print("CHECKPOINT EVALUATION SUMMARY")
print("=" * 80)

print()

print(
    f"Vocabulary Size       : "
    f"{config.vocab_size:,}"
)

print(
    f"Context Length        : "
    f"{config.max_position_embeddings:,}"
)

print(
    f"Hidden Size           : "
    f"{config.hidden_size:,}"
)

print(
    f"Transformer Layers    : "
    f"{config.num_layers}"
)

print(
    f"Attention Heads       : "
    f"{config.num_attention_heads}"
)

print(
    f"Intermediate Size     : "
    f"{config.intermediate_size:,}"
)

print(
    f"Total Parameters      : "
    f"{total_parameters:,}"
)

print(
    f"Model Device          : "
    f"{next(model.parameters()).device}"
)

print()

if (
    "eval_loss" in globals()
    and
    "eval_perplexity" in globals()
):

    print(
        f"Held-Out Loss         : "
        f"{eval_loss:.6f}"
    )

    print(
        f"Held-Out Perplexity   : "
        f"{eval_perplexity:.4f}"
    )

else:

    print(
        "Held-Out Loss         : unavailable"
    )

    print(
        "Held-Out Perplexity   : unavailable"
    )

print()

if len(steps) > 0:

    print(
        f"Training Log          : available"
    )

    print(
        f"Logged Training Steps : "
        f"{len(steps):,}"
    )

else:

    print(
        "Training Log          : unavailable"
    )

    print(
        "Training-log analysis : skipped safely"
    )

print()
print("=" * 80)
print("CHECKPOINT EVALUATION COMPLETE")
print("=" * 80)

CHECKPOINT EVALUATION SUMMARY

Vocabulary Size       : 32,000
Context Length        : 512
Hidden Size           : 768
Transformer Layers    : 12
Attention Heads       : 12
Intermediate Size     : 3,072
Total Parameters      : 110,025,216
Model Device          : cuda:0

Held-Out Loss         : 10.394073
Held-Out Perplexity   : 32665.4559

Training Log          : unavailable
Training-log analysis : skipped safely

CHECKPOINT EVALUATION COMPLETE


In [ ]:
print("=" * 80)
print("CHECKPOINT LOAD DIAGNOSTIC")
print("=" * 80)

print("Checkpoint keys:")

if isinstance(checkpoint, dict):
    for key in checkpoint.keys():
        print(" ", key)

print()

if isinstance(checkpoint, dict):

    for key in [
        "step",
        "global_step",
        "epoch",
        "loss",
        "train_loss",
        "best_loss",
    ]:

        if key in checkpoint:
            print(
                f"{key:<20}: {checkpoint[key]}"
            )

print("=" * 80)

CHECKPOINT LOAD DIAGNOSTIC
Checkpoint keys:
  checkpoint_version
  created_at
  epoch
  global_step
  best_loss
  train_loss
  val_loss
  model_state_dict
  optimizer_state_dict
  scheduler_state_dict
  random_states
  config
  extra
  mygpt2_training_manifest
  mygpt2_checkpoint_version
  mygpt2_checkpoint_saved_at

global_step         : 213000
epoch               : 0
train_loss          : 4.340525150299072
best_loss           : None


In [ ]:
print("=" * 80)
print("MODEL WEIGHT DIAGNOSTIC")
print("=" * 80)

for name, parameter in model.named_parameters():

    if parameter.numel() == 0:
        continue

    data = parameter.detach().float()

    print(
        f"{name:<55} "
        f"mean={data.mean().item():+.6f}  "
        f"std={data.std().item():.6f}"
    )

    # Only inspect first few
    if "layers.1" in name:
        break

print("=" * 80)

MODEL WEIGHT DIAGNOSTIC
embeddings.token_embeddings.weight                      mean=+0.000003  std=0.020001
embeddings.position_embeddings.weight                   mean=-0.000037  std=0.020029
blocks.0.ln1.weight                                     mean=+1.000000  std=0.000000
blocks.0.ln1.bias                                       mean=+0.000000  std=0.000000
blocks.0.attention.qkv_proj.weight                      mean=-0.000016  std=0.020006
blocks.0.attention.qkv_proj.bias                        mean=+0.000000  std=0.000000
blocks.0.attention.out_proj.weight                      mean=+0.000002  std=0.004080
blocks.0.attention.out_proj.bias                        mean=+0.000000  std=0.000000
blocks.0.ln2.weight                                     mean=+1.000000  std=0.000000
blocks.0.ln2.bias                                       mean=+0.000000  std=0.000000
blocks.0.mlp.fc1.weight                                 mean=-0.000005  std=0.020004
blocks.0.mlp.fc1.bias                    

In [ ]:
# ============================================================
# MANUAL CROSS-ENTROPY SANITY CHECK
# ============================================================

import torch
import torch.nn.functional as F
import math

model.eval()

text = (
    "Machine learning is a field of artificial intelligence "
    "that allows computers to learn patterns from data."
)

tokens = tokenizer.encode(text)

if isinstance(tokens, dict):
    tokens = tokens["input_ids"]

if not isinstance(tokens, torch.Tensor):
    tokens = torch.tensor(
        tokens,
        dtype=torch.long,
    )

tokens = tokens.reshape(-1).to(
    next(model.parameters()).device
)

input_ids = tokens[:-1].unsqueeze(0)
targets = tokens[1:].unsqueeze(0)

print("Input shape :", input_ids.shape)
print("Target shape:", targets.shape)

with torch.no_grad():

    output = model(
        input_ids=input_ids
    )

if hasattr(output, "logits"):
    logits = output.logits

elif isinstance(output, dict):
    logits = output["logits"]

elif isinstance(output, (tuple, list)):
    logits = output[0]

else:
    logits = output


loss = F.cross_entropy(
    logits.reshape(-1, logits.size(-1)),
    targets.reshape(-1),
)

ppl = math.exp(loss.item())

print()
print("=" * 80)
print("MANUAL LOSS TEST")
print("=" * 80)

print(f"Loss       : {loss.item():.6f}")
print(f"Perplexity : {ppl:.4f}")

print("=" * 80)

Input shape : torch.Size([1, 18])
Target shape: torch.Size([1, 18])

MANUAL LOSS TEST
Loss       : 10.621406
Perplexity : 41003.2086


In [ ]:
# ============================================================
# CHECKPOINT vs LOADED MODEL WEIGHT DIAGNOSTIC
# ============================================================

import torch

print("=" * 80)
print("CHECKPOINT vs LOADED MODEL")
print("=" * 80)

checkpoint_state = checkpoint["model_state_dict"]
loaded_state = model.state_dict()

print()
print(
    f"Checkpoint tensors : {len(checkpoint_state):,}"
)

print(
    f"Model tensors      : {len(loaded_state):,}"
)

print()

# ------------------------------------------------------------
# Key comparison
# ------------------------------------------------------------

checkpoint_keys = set(checkpoint_state.keys())
model_keys = set(loaded_state.keys())

missing_in_model = checkpoint_keys - model_keys
missing_in_checkpoint = model_keys - checkpoint_keys

print(
    f"Checkpoint-only keys : {len(missing_in_model)}"
)

print(
    f"Model-only keys      : {len(missing_in_checkpoint)}"
)

if missing_in_model:

    print()
    print("Example checkpoint-only keys:")

    for key in list(missing_in_model)[:10]:
        print(" ", key)

if missing_in_checkpoint:

    print()
    print("Example model-only keys:")

    for key in list(missing_in_checkpoint)[:10]:
        print(" ", key)


# ------------------------------------------------------------
# Compare matching tensors
# ------------------------------------------------------------

print()
print("-" * 80)
print("TENSOR COMPARISON")
print("-" * 80)

matching = 0
different = 0

max_difference = 0.0

for name in checkpoint_state:

    if name not in loaded_state:
        continue

    checkpoint_tensor = (
        checkpoint_state[name]
        .detach()
        .cpu()
        .float()
    )

    loaded_tensor = (
        loaded_state[name]
        .detach()
        .cpu()
        .float()
    )

    difference = (
        checkpoint_tensor - loaded_tensor
    ).abs().max().item()

    max_difference = max(
        max_difference,
        difference,
    )

    if difference == 0:
        matching += 1
    else:
        different += 1


print()
print(
    f"Exactly matching tensors : {matching}"
)

print(
    f"Different tensors        : {different}"
)

print(
    f"Maximum difference       : {max_difference:.10f}"
)


# ------------------------------------------------------------
# Inspect checkpoint weights DIRECTLY
# ------------------------------------------------------------

print()
print("-" * 80)
print("CHECKPOINT WEIGHT STATISTICS")
print("-" * 80)

important_weights = [

    "embeddings.token_embeddings.weight",

    "embeddings.position_embeddings.weight",

    "blocks.0.attention.qkv_proj.weight",

    "blocks.0.attention.out_proj.weight",

    "blocks.0.mlp.fc1.weight",

    "blocks.0.mlp.fc2.weight",

    "blocks.11.attention.qkv_proj.weight",

    "blocks.11.mlp.fc1.weight",

    "final_layer_norm.weight",
]


for name in important_weights:

    if name not in checkpoint_state:
        continue

    tensor = (
        checkpoint_state[name]
        .detach()
        .float()
        .cpu()
    )

    print(
        f"{name:<55} "
        f"mean={tensor.mean().item():+.8f} "
        f"std={tensor.std().item():.8f} "
        f"min={tensor.min().item():+.8f} "
        f"max={tensor.max().item():+.8f}"
    )


print()
print("=" * 80)

CHECKPOINT vs LOADED MODEL

Checkpoint tensors : 149
Model tensors      : 149

Checkpoint-only keys : 0
Model-only keys      : 0

--------------------------------------------------------------------------------
TENSOR COMPARISON
--------------------------------------------------------------------------------

Exactly matching tensors : 0
Different tensors        : 149
Maximum difference       : 8.0244693756

--------------------------------------------------------------------------------
CHECKPOINT WEIGHT STATISTICS
--------------------------------------------------------------------------------
embeddings.token_embeddings.weight                      mean=-0.06718025 std=0.14803497 min=-1.00774014 max=+1.88719451
embeddings.position_embeddings.weight                   mean=+0.02871486 std=0.04529823 min=-0.39192551 max=+0.33490652
blocks.0.attention.qkv_proj.weight                      mean=+0.00024181 std=0.11274696 min=-1.16984344 max=+1.16189301
blocks.0.attention.out_proj.weight   